[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/slam-handbook-python/blob/main/part1_foundations/ch02_state_representations/03_continuous_trajectories.ipynb)

# Chapter 2: 連続時間軌道

**SLAM Handbook Section 2.2** — スプラインとガウス過程による連続時間軌道の表現。

## このNotebookで学ぶこと

1. **離散 vs 連続時間** — なぜ連続時間軌道が重要か
2. **スプライン補間** — 基底関数による軌道の表現（ベクトル空間 + Lie群上）
3. **ガウス過程 (GP) 回帰** — SDE駆動のGPモーションプライア
4. **GPの逆カーネル行列** — なぜスパースになるか（マルコフ性）
5. **連続時間SLAMの利点** — motion-distortedセンサへの対応

### なぜ重要か？
- LiDAR/カメラのスキャン中にロボットが動く → **motion distortion**
- IMU (1000Hz) とカメラ (30Hz) のタイムスタンプが異なる → **非同期センサ融合**
- 離散ポーズを計測ごとに追加するとグラフが肥大化 → **制御点ベースで効率化**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm, block_diag
from scipy.interpolate import BSpline

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 3.1 離散 vs 連続時間

### 離散時間アプローチの問題
Ch01では、計測ごとにポーズ変数を追加しました。これだと：
- IMU 1000Hz → 1秒間に1000ポーズ変数が必要
- LiDARの各点が異なる時刻に取得される → 各点に個別ポーズが必要

### 連続時間アプローチ（SLAM Handbook Eq. 2.36）
軌道を少数の **制御点** $\mathbf{c}_k$ と **基底関数** $\Psi_k(t)$ で表現：

$$\mathbf{p}(t) = \sum_{k=1}^{K} \Psi_k(t) \, \mathbf{c}_k$$

→ 任意の時刻 $t$ でのポーズをクエリ可能。計測数 >> 制御点数。

In [ ]:
# =============================================================
# 離散 vs 連続時間の比較デモ
# =============================================================
np.random.seed(42)

# 真の軌道: 滑らかな2D曲線
t_true = np.linspace(0, 5, 500)
x_true = 2 * np.sin(0.8 * t_true) + 0.5 * t_true
y_true = 1.5 * np.cos(0.5 * t_true) + 0.3 * t_true

# 離散計測（非同期）
t_cam = np.arange(0, 5, 0.5)     # カメラ: 2Hz
t_imu = np.arange(0, 5, 0.02)    # IMU: 50Hz
t_lidar = np.arange(0, 5, 0.1)   # LiDAR: 10Hz

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 左: 離散時間（各計測にポーズ変数）
ax = axes[0]
ax.plot(x_true, y_true, 'k-', alpha=0.3, label='True trajectory')
# 各センサのタイムスタンプでポーズを追加
for t_arr, color, label, ms in [(t_cam, 'red', 'Camera (2Hz)', 8),
                                  (t_lidar, 'blue', 'LiDAR (10Hz)', 5),
                                  (t_imu, 'green', 'IMU (50Hz)', 2)]:
    x_interp = np.interp(t_arr, t_true, x_true)
    y_interp = np.interp(t_arr, t_true, y_true)
    ax.plot(x_interp, y_interp, 'o', color=color, markersize=ms, label=label, alpha=0.6)
ax.set_title(f'Discrete: {len(t_cam)+len(t_imu)+len(t_lidar)} pose variables needed',
             fontweight='bold')
ax.legend(fontsize=9); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

# 右: 連続時間（少数の制御点）
ax = axes[1]
ax.plot(x_true, y_true, 'k-', alpha=0.3, label='True trajectory')
# 制御点（少数）
t_ctrl = np.linspace(0, 5, 8)
x_ctrl = np.interp(t_ctrl, t_true, x_true)
y_ctrl = np.interp(t_ctrl, t_true, y_true)
ax.plot(x_ctrl, y_ctrl, 'ks', markersize=12, label=f'Control points ({len(t_ctrl)})')
# 補間軌道
ax.plot(np.interp(t_true, t_true, x_true), np.interp(t_true, t_true, y_true),
        'b-', alpha=0.5, label='Interpolated')
ax.set_title(f'Continuous-time: only {len(t_ctrl)} control points',
             fontweight='bold')
ax.legend(fontsize=9); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"離散: {len(t_cam)+len(t_imu)+len(t_lidar)} 変数 vs 連続: {len(t_ctrl)} 変数")

## 3.2 スプライン基底関数

SLAM Handbook Section 2.2.1: 基底関数には **local support** を持つスプラインが使われます。

### 線形スプライン (SLAM Handbook Eq. 2.56-2.57)

最もシンプルな場合、制御点間を線形補間：

$$\mathbf{p}(t) = (1 - \alpha_k(t)) \, \mathbf{p}_{k-1} + \alpha_k(t) \, \mathbf{p}_k, \quad \alpha_k(t) = \frac{t - (k-1)T}{T}$$

ここで $(k-1)T \leq t < kT$。

### 累積形式（Lie群への拡張に必要）

$$\mathbf{p}(t) = \mathbf{p}_{k-1} + \psi_k^c(t) (\mathbf{p}_k - \mathbf{p}_{k-1})$$

Lie群では差分をLogで取り、Expで合成（SLAM Handbook Eq. 2.61）：

$$\mathbf{T}(t) = \text{Exp}\left(\psi_k^c(t) \, \text{Log}(\mathbf{T}_k \mathbf{T}_{k-1}^{-1})\right) \cdot \mathbf{T}_{k-1}$$

In [ ]:
# =============================================================
# スプライン基底関数の可視化 (SLAM Handbook Fig. 2.5)
# =============================================================

K = 7  # 制御点数
T_interval = 1.0  # 制御点間隔
t_dense = np.linspace(0, (K-1)*T_interval, 500)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# --- 通常の線形基底関数 ψ_k(t) ---
ax = axes[0]
colors = plt.cm.tab10(np.linspace(0, 1, K))
for k in range(K):
    psi = np.zeros_like(t_dense)
    center = k * T_interval
    for i, t in enumerate(t_dense):
        if k == 0:
            if 0 <= t < T_interval:
                psi[i] = 1 - t / T_interval
        elif k == K - 1:
            if (K-2)*T_interval <= t <= (K-1)*T_interval:
                psi[i] = (t - (K-2)*T_interval) / T_interval
        else:
            if (k-1)*T_interval <= t < k*T_interval:
                psi[i] = (t - (k-1)*T_interval) / T_interval
            elif k*T_interval <= t < (k+1)*T_interval:
                psi[i] = 1 - (t - k*T_interval) / T_interval
    ax.plot(t_dense, psi, color=colors[k], linewidth=2, label=f'ψ_{k+1}(t)')

ax.set_ylabel('ψ_k(t)')
ax.set_title('Linear Spline Basis Functions (SLAM Handbook Fig. 2.5 top)', fontweight='bold')
ax.legend(ncol=K, fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)

# --- 累積基底関数 ψ^c_k(t) ---
ax = axes[1]
for k in range(K):
    psi_c = np.zeros_like(t_dense)
    for i, t in enumerate(t_dense):
        if k == 0:
            psi_c[i] = 1.0  # always 1
        elif k == K - 1:
            if t >= (K-2)*T_interval:
                psi_c[i] = min((t - (K-2)*T_interval) / T_interval, 1.0)
        else:
            if t < (k-1)*T_interval:
                psi_c[i] = 0
            elif t < k*T_interval:
                psi_c[i] = (t - (k-1)*T_interval) / T_interval
            else:
                psi_c[i] = 1.0
    ax.plot(t_dense, psi_c, color=colors[k], linewidth=2, label=f'ψ^c_{k+1}(t)')

ax.set_xlabel('t')
ax.set_ylabel('ψ^c_k(t)')
ax.set_title('Cumulative Basis Functions (Fig. 2.5 bottom)', fontweight='bold')
ax.legend(ncol=K, fontsize=8, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("→ 通常形式: 各時刻で高々2つの基底のみ非ゼロ (local support)")
print("→ 累積形式: Lie群上のスプラインに必要（差分の合成に使う）")

## 3.3 SE(2)上のスプライン補間

Lie群上ではベクトルの足し算を行列の積に、差分をLogに置き換えます。

**線形スプライン on SE(2)** (SLAM Handbook Eq. 2.61-2.62)：

$$\mathbf{T}(t) = \left(\mathbf{T}_k \mathbf{T}_{k-1}^{-1}\right)^{\alpha_k(t)} \mathbf{T}_{k-1}$$

ここで行列のべき乗は $\mathbf{X}^{\alpha} = \text{Exp}(\alpha \, \text{Log}(\mathbf{X}))$ で計算します。

In [ ]:
# =============================================================
# SE(2)上のスプライン補間
# =============================================================

# SE(2) ユーティリティ（前のNotebookから再掲）
def se2_from_xyt(x, y, th):
    c, s = np.cos(th), np.sin(th)
    return np.array([[c, -s, x], [s, c, y], [0, 0, 1]])

def se2_inv(T):
    R = T[:2, :2]; t = T[:2, 2]
    Ti = np.eye(3); Ti[:2, :2] = R.T; Ti[:2, 2] = -R.T @ t
    return Ti

def se2_log(T):
    """SE(2) → R^3"""
    th = np.arctan2(T[1,0], T[0,0])
    t = T[:2, 2]
    if abs(th) < 1e-10:
        return np.array([t[0], t[1], th])
    c, s = np.cos(th), np.sin(th)
    V = np.array([[s/th, -(1-c)/th], [(1-c)/th, s/th]])
    rho = np.linalg.solve(V, t)
    return np.array([rho[0], rho[1], th])

def se2_exp(xi):
    """R^3 → SE(2)"""
    rho, th = xi[:2], xi[2]
    if abs(th) < 1e-10:
        return np.array([[1, 0, rho[0]], [0, 1, rho[1]], [0, 0, 1]])
    c, s = np.cos(th), np.sin(th)
    V = np.array([[s/th, -(1-c)/th], [(1-c)/th, s/th]])
    t = V @ rho
    T = np.eye(3); T[:2, :2] = np.array([[c, -s], [s, c]]); T[:2, 2] = t
    return T

def slerp_se2(T0, T1, alpha):
    """SE(2)上のSLERP: T(α) = Exp(α·Log(T1·T0^{-1})) · T0"""
    delta = se2_log(T1 @ se2_inv(T0))
    return se2_exp(alpha * delta) @ T0

# 制御点を定義（円弧状の軌跡）
control_poses = [
    se2_from_xyt(0, 0, 0),
    se2_from_xyt(2, 0.5, np.pi/6),
    se2_from_xyt(3.5, 2, np.pi/3),
    se2_from_xyt(3, 4, 2*np.pi/3),
    se2_from_xyt(1, 4.5, np.pi),
    se2_from_xyt(-0.5, 3, -2*np.pi/3),
]
K = len(control_poses)

# 補間
n_interp = 200
t_interp = np.linspace(0, K-1, n_interp)

poses_interp = []
for t in t_interp:
    k = min(int(t), K-2)
    alpha = t - k
    T_interp = slerp_se2(control_poses[k], control_poses[k+1], alpha)
    poses_interp.append(T_interp)

# 可視化
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# 補間軌跡
xs = [T[0, 2] for T in poses_interp]
ys = [T[1, 2] for T in poses_interp]
ax.plot(xs, ys, 'b-', linewidth=2, label='Interpolated trajectory', alpha=0.7)

# 補間ポーズの方向（間引き）
for i in range(0, len(poses_interp), 10):
    T = poses_interp[i]
    x, y = T[0, 2], T[1, 2]
    th = np.arctan2(T[1, 0], T[0, 0])
    ax.arrow(x, y, 0.15*np.cos(th), 0.15*np.sin(th),
             head_width=0.05, fc='blue', ec='blue', alpha=0.4)

# 制御点
for i, T in enumerate(control_poses):
    x, y = T[0, 2], T[1, 2]
    th = np.arctan2(T[1, 0], T[0, 0])
    ax.plot(x, y, 'rs', markersize=12, zorder=5)
    ax.arrow(x, y, 0.3*np.cos(th), 0.3*np.sin(th),
             head_width=0.1, fc='red', ec='red', zorder=5)
    ax.text(x+0.15, y+0.2, f'T{i}', fontsize=11, color='red', fontweight='bold')

ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=12)
ax.set_title('SE(2) Linear Spline Interpolation\n'
             'T(t) = Exp(α·Log(T_k·T_{k-1}⁻¹)) · T_{k-1}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("→ 制御点間をLie群上で滑らかに補間（SLERP）")
print("→ 任意の時刻tでのポーズT(t)をクエリ可能")

## 3.4 ガウス過程 (GP) モーションプライア

SLAM Handbook Section 2.2.3: SDEから導かれるGPカーネルは、逆カーネル行列が **ブロック三重対角** になるため、Factor Graphがスパースに保たれます。

### Constant-Velocity Prior (SLAM Handbook Eq. 2.52)

状態 $\mathbf{x}(t) = [\mathbf{p}(t)^\top, \mathbf{v}(t)^\top]^\top$ （位置+速度）のSDE：

$$\dot{\mathbf{x}}(t) = \begin{bmatrix} \mathbf{0} & \mathbf{I} \\ \mathbf{0} & \mathbf{0} \end{bmatrix} \mathbf{x}(t) + \begin{bmatrix} \mathbf{0} \\ \mathbf{I} \end{bmatrix} \mathbf{w}(t)$$

遷移関数: $\boldsymbol{\Phi}(\Delta t) = \exp(\mathbf{A} \Delta t) = \begin{bmatrix} \mathbf{I} & \Delta t \, \mathbf{I} \\ \mathbf{0} & \mathbf{I} \end{bmatrix}$

**ポイント**: $\boldsymbol{\Phi}^{-1}$ がブロック2重対角 → $\mathbf{K}^{-1}$ がブロック三重対角 → Factor Graphがスパース

In [ ]:
# =============================================================
# GP Motion Prior: Constant-Velocity model
# =============================================================

def gp_constant_velocity(t_knots, q_c, dim=1):
    """Constant-velocity GPモーションプライアの構築
    
    Parameters
    ----------
    t_knots : array — 制御点の時刻
    q_c : float — プロセスノイズの強度
    dim : int — 空間次元
    
    Returns
    -------
    Phi_inv : 遷移行列の逆 (block bidiagonal)
    Q_inv : プロセスノイズの逆 (block diagonal)
    """
    M = len(t_knots)
    state_dim = 2 * dim  # [position, velocity]
    
    # 遷移行列 Φ(Δt) と プロセスノイズ Q(Δt) の計算
    Phi_list = []
    Q_list = []
    
    for i in range(M - 1):
        dt = t_knots[i+1] - t_knots[i]
        
        # Φ(dt) = [[I, dt*I], [0, I]]
        Phi = np.eye(state_dim)
        Phi[:dim, dim:] = dt * np.eye(dim)
        
        # Q(dt) = q_c * [[dt³/3, dt²/2], [dt²/2, dt]] ⊗ I_dim
        Q_scalar = q_c * np.array([
            [dt**3/3, dt**2/2],
            [dt**2/2, dt]
        ])
        Q = np.kron(Q_scalar, np.eye(dim))
        
        Phi_list.append(Phi)
        Q_list.append(Q)
    
    return Phi_list, Q_list

# 1Dの例
t_knots = np.array([0.0, 1.0, 2.0, 3.0, 4.0, 5.0])
M = len(t_knots)
q_c = 1.0  # プロセスノイズ強度
dim = 1
state_dim = 2 * dim

Phi_list, Q_list = gp_constant_velocity(t_knots, q_c, dim)

# Φ^{-1} のスパース構造を構築 (SLAM Handbook Eq. 2.51)
n_total = state_dim * M
Phi_inv_full = np.eye(n_total)
for i in range(M - 1):
    si = state_dim * (i + 1)
    Phi_inv_full[si:si+state_dim, si-state_dim:si] = -Phi_list[i]

# K^{-1} = Φ^{-T} Q^{-1} Φ^{-1}
Q_inv_full = np.zeros((n_total, n_total))
# 初期状態の事前情報
Q_inv_full[:state_dim, :state_dim] = np.eye(state_dim) * 100  # 強いprior
for i in range(M - 1):
    si = state_dim * (i + 1)
    Q_inv_full[si:si+state_dim, si:si+state_dim] = np.linalg.inv(Q_list[i])

K_inv = Phi_inv_full.T @ Q_inv_full @ Phi_inv_full

# 可視化
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].spy(Phi_inv_full, markersize=8, color='steelblue')
axes[0].set_title('$\\Phi^{-1}$ (Block bidiagonal)', fontweight='bold')

axes[1].spy(Q_inv_full, markersize=8, color='steelblue')
axes[1].set_title('$Q^{-1}$ (Block diagonal)', fontweight='bold')

axes[2].spy(K_inv, markersize=8, color='steelblue')
axes[2].set_title('$K^{-1} = \\Phi^{-T}Q^{-1}\\Phi^{-1}$\n(Block tridiagonal!)', fontweight='bold')

plt.tight_layout()
plt.show()

print("→ K^{-1}がブロック三重対角 = Factor Graphで隣接ノード間のファクターのみ")
print("→ マルコフ性: x_{k+1} は x_k のみに依存（x_{k-1} 以前と条件付き独立）")

In [ ]:
# =============================================================
# GP回帰: ノイズ付き位置計測から連続軌道を推定
# =============================================================
np.random.seed(42)

# 真の軌道: 正弦波的な位置
def true_position(t):
    return 2 * np.sin(0.8 * t) + 0.3 * t

def true_velocity(t):
    return 2 * 0.8 * np.cos(0.8 * t) + 0.3

# ノイズ付き位置計測（非等間隔！）
t_meas = np.sort(np.random.uniform(0, 5, 15))
sigma_meas = 0.3
z_meas = true_position(t_meas) + np.random.normal(0, sigma_meas, len(t_meas))

# GP推定: 制御点でのstate = [pos, vel] を推定
# 正規方程式: (K^{-1} + A^T Σ^{-1} A) x* = A^T Σ^{-1} z
n_total = state_dim * M

# 計測のヤコビアンとRHS
ATA = np.zeros((n_total, n_total))
ATb = np.zeros(n_total)

for z, tm in zip(z_meas, t_meas):
    # tmがどのセグメント [t_k, t_{k+1}) に入るか
    k = np.searchsorted(t_knots, tm, side='right') - 1
    k = np.clip(k, 0, M - 2)
    
    dt = tm - t_knots[k]
    # 遷移: x(tm) = Φ(dt) @ x(tk)
    Phi_dt = np.eye(state_dim)
    Phi_dt[0, 1] = dt
    
    # 計測は位置のみ: H = [1, 0] @ Φ(dt) で x_k に関するヤコビアン
    H_local = np.array([[1, 0]]) @ Phi_dt  # 1×2
    
    si = state_dim * k
    H_full = np.zeros((1, n_total))
    H_full[0, si:si+state_dim] = H_local
    
    # 残差
    # 予測は x_k に依存するので、最小二乗に蓄積
    ATA += H_full.T @ H_full / sigma_meas**2
    ATb += H_full.T @ np.array([z]) / sigma_meas**2

# 事後分布: (K^{-1} + A^T Σ^{-1} A)^{-1}
Lambda = K_inv + ATA
x_est = np.linalg.solve(Lambda, ATb)
cov_est = np.linalg.inv(Lambda)

# 結果の可視化
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

t_dense = np.linspace(0, 5, 300)

# --- 位置 ---
ax = axes[0]
ax.plot(t_dense, true_position(t_dense), 'k-', linewidth=2, label='True position')
ax.plot(t_meas, z_meas, 'r.', markersize=10, label=f'Measurements (σ={sigma_meas})')

# GP推定の位置を補間
pos_est = []
pos_std = []
for t in t_dense:
    k = np.searchsorted(t_knots, t, side='right') - 1
    k = np.clip(k, 0, M - 2)
    dt = t - t_knots[k]
    Phi_dt = np.eye(state_dim)
    Phi_dt[0, 1] = dt
    
    si = state_dim * k
    x_k = x_est[si:si+state_dim]
    pos_est.append((Phi_dt @ x_k)[0])
    
    # 不確かさ
    H_interp = np.zeros((1, n_total))
    H_interp[0, si:si+state_dim] = np.array([1, 0]) @ Phi_dt
    var_pos = H_interp @ cov_est @ H_interp.T
    pos_std.append(np.sqrt(max(var_pos[0, 0], 0)))

pos_est = np.array(pos_est)
pos_std = np.array(pos_std)

ax.plot(t_dense, pos_est, 'b-', linewidth=2, label='GP estimate')
ax.fill_between(t_dense, pos_est - 2*pos_std, pos_est + 2*pos_std,
                alpha=0.2, color='blue', label='2σ confidence')
ax.plot(t_knots, x_est[::2], 'bs', markersize=8, label='Control points')
ax.set_ylabel('Position')
ax.legend(fontsize=9)
ax.set_title('GP Continuous-Time Trajectory Estimation', fontweight='bold')
ax.grid(True, alpha=0.3)

# --- 速度 ---
ax = axes[1]
ax.plot(t_dense, true_velocity(t_dense), 'k-', linewidth=2, label='True velocity')

vel_est = []
for t in t_dense:
    k = np.searchsorted(t_knots, t, side='right') - 1
    k = np.clip(k, 0, M - 2)
    dt = t - t_knots[k]
    Phi_dt = np.eye(state_dim)
    Phi_dt[0, 1] = dt
    si = state_dim * k
    x_k = x_est[si:si+state_dim]
    vel_est.append((Phi_dt @ x_k)[1])

ax.plot(t_dense, vel_est, 'b-', linewidth=2, label='GP estimated velocity')
ax.plot(t_knots, x_est[1::2], 'bs', markersize=8, label='Control points (vel)')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Velocity')
ax.legend(fontsize=9)
ax.set_title('Velocity (derivative of trajectory)', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("→ 位置計測のみから速度も推定可能（GPの微分性質）")
print("→ 非等間隔の計測でも自然に扱える")
print("→ 不確かさは計測がない区間で大きくなる")

## 3.5 演習問題

### 演習1: 3次Bスプライン
線形スプラインをcubic B-splineに変更して、より滑らかな軌道を得てみましょう。`scipy.interpolate.BSpline` が使えます。

### 演習2: プロセスノイズ $q_c$ の影響
GP回帰で `q_c` を 0.01, 1.0, 100.0 に変えて結果を比較してください。$q_c$ が小さいと軌道は滑らかになりますが、計測への追従が悪くなります。

### 演習3: SE(2)上のGP
1Dの位置のGPを SE(2) 上に拡張してみましょう。状態は $[\mathbf{T}_k, \dot{\boldsymbol{\xi}}_k]$ （ポーズ+Lie代数上の速度）になります。SLAM Handbook Section 2.2.4.2 を参考にしてください。

### 演習4: Motion Distortion補正
LiDARスキャン中にロボットが動く場面をシミュレートし、連続時間軌道を使って各点のタイムスタンプに対応するポーズで補正してみましょう。

## まとめ

| 手法 | 利点 | 欠点 |
|---|---|---|
| **離散時間** | シンプル | 計測数=変数数、非同期困難 |
| **スプライン** | 滑らか、local support | 基底関数数の選択が必要 |
| **GP (SDE)** | 自動的に滑らか、補間+微分 | SDEの設計が必要 |

- **連続時間軌道** により、任意時刻のポーズをクエリ可能
- **スプライン**: 制御点+基底関数で軌道を表現。Lie群上では累積形式を使用
- **ガウス過程**: SDE駆動のGPカーネルは $\mathbf{K}^{-1}$ がスパース → Factor Graphと相性抜群
- 位置計測のみから **速度も推定可能**（GPの微分性質）
- **STEAM** (Simultaneous Trajectory Estimation and Mapping) の基礎

### Chapter 2 完了！
→ Ch03: 外れ値対策（RANSAC, Robust Cost Functions, GNC）